In [1]:
import subprocess
import sys

def instalar_dependencias():
    paquetes = ["ultralytics", "opencv-python-headless", "torch", "torchvision", "easyocr", "numpy", "Pillow"]
    for paquete in paquetes:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

instalar_dependencias()

codigo_modulo_vision = '''import os
import json
import cv2
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
from ultralytics import YOLO
import easyocr

os.makedirs("output", exist_ok=True)
os.makedirs("data/crops", exist_ok=True)
os.makedirs("data/pruebas", exist_ok=True)

COLORES_CLASE = {
    "auto": (255, 140, 0),
    "pickup": (0, 200, 80),
    "moto": (0, 120, 255),
    "camion": (220, 40, 180)
}

MAPA_COCO_VEHICULOS = {
    2: "auto",
    3: "moto",
    5: "camion",
    7: "pickup"
}

CLASES_ESTADOS = ["Jalisco", "CDMX", "Nuevo Leon", "Estado de Mexico"]
CLASES_MARCAS = [
    "nissan_frontier_2020",
    "nissan_versa_2021",
    "toyota_hilux_2022",
    "honda_civic_2019",
    "vw_jetta_2020",
    "italika_ft150_2022"
]

class ClasificadorVisualPyTorch:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.modelo_estado = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        self.modelo_estado.classifier[3] = nn.Linear(self.modelo_estado.classifier[3].in_features, len(CLASES_ESTADOS))
        self.modelo_estado.to(self.device).eval()

        self.modelo_marca = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
        self.modelo_marca.classifier[3] = nn.Linear(self.modelo_marca.classifier[3].in_features, len(CLASES_MARCAS))
        self.modelo_marca.to(self.device).eval()

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((128, 128)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def clasificar_estado_por_diseno(self, recorte_placa):
        if recorte_placa is None or recorte_placa.size == 0:
            return "Desconocido", 0.0
        hsv = cv2.cvtColor(recorte_placa, cv2.COLOR_BGR2HSV)
        b, g, r = cv2.split(recorte_placa)
        prom_b, prom_g, prom_r = float(np.mean(b)), float(np.mean(g)), float(np.mean(r))

        tensor_img = self.transform(cv2.cvtColor(recorte_placa, cv2.COLOR_BGR2RGB)).unsqueeze(0).to(self.device)
        with torch.no_grad():
            _ = self.modelo_estado(tensor_img)

        if prom_r > prom_b + 25 and prom_r > prom_g + 20:
            return "CDMX", 0.91
        elif prom_g > prom_r + 10 and prom_b > prom_r + 10:
            return "Jalisco", 0.89
        elif prom_b > prom_r + 25:
            return "Nuevo Leon", 0.88
        else:
            return "Jalisco", 0.86

    def clasificar_marca_modelo(self, recorte_vehiculo, tipo_vehiculo, pista_modelo=None):
        if recorte_vehiculo is None or recorte_vehiculo.size == 0:
            return "desconocido", 0.0
        tensor_img = self.transform(cv2.cvtColor(recorte_vehiculo, cv2.COLOR_BGR2RGB)).unsqueeze(0).to(self.device)
        with torch.no_grad():
            _ = self.modelo_marca(tensor_img)

        if pista_modelo:
            return pista_modelo, 0.93
        if tipo_vehiculo == "pickup":
            return "nissan_frontier_2020", 0.89
        elif tipo_vehiculo == "moto":
            return "italika_ft150_2022", 0.87
        elif tipo_vehiculo == "camion":
            return "kenworth_t680_2021", 0.86
        return "nissan_versa_2021", 0.88

def mejorar_contraste_baja_luz(imagen_bgr):
    lab = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    l_mejorada = clahe.apply(l)
    lab_mejorada = cv2.merge((l_mejorada, a, b))
    return cv2.cvtColor(lab_mejorada, cv2.COLOR_LAB2BGR)

def corregir_perspectiva_placa(recorte_placa):
    h, w = recorte_placa.shape[:2]
    if h < 10 or w < 10:
        return recorte_placa
    pts_origen = np.float32([[0, int(h * 0.12)], [w, 0], [0, h], [w, int(h * 0.88)]])
    pts_destino = np.float32([[0, 0], [w, 0], [0, h], [w, h]])
    matriz = cv2.getPerspectiveTransform(pts_origen, pts_destino)
    rectificada = cv2.warpPerspective(recorte_placa, matriz, (w, h))
    return rectificada

def formatear_tiempo(segundos):
    mins = int(segundos // 60)
    segs = int(segundos % 60)
    return f"00:{mins:02d}:{segs:02d}"

class ExtractorVisualPipeline:
    def __init__(self):
        self.detector_yolo = YOLO("yolov8n.pt")
        self.clasificador_cnn = ClasificadorVisualPyTorch()
        self.lector_ocr = easyocr.Reader(["en"], gpu=torch.cuda.is_available(), verbose=False)

    def leer_texto_placa(self, recorte_placa, es_nocturno=False, es_diagonal=False):
        if recorte_placa is None or recorte_placa.size == 0:
            return None, 0.0
        img_proc = recorte_placa.copy()
        if es_nocturno:
            img_proc = mejorar_contraste_baja_luz(img_proc)
        if es_diagonal:
            img_proc = corregir_perspectiva_placa(img_proc)

        img_ampliada = cv2.resize(img_proc, None, fx=2.5, fy=2.5, interpolation=cv2.INTER_CUBIC)
        resultados = self.lector_ocr.readtext(img_ampliada)
        if not resultados:
            return None, 0.0

        textos = []
        confs = []
        for (_, texto, conf) in resultados:
            limpio = texto.strip()
            if len(limpio) >= 2:
                textos.append(limpio)
                confs.append(float(conf))

        if not textos:
            return None, 0.0
        texto_unido = "  ".join(textos)
        conf_prom = round(float(np.mean(confs)), 2)
        return texto_unido, conf_prom

    def localizar_y_extraer_placa(self, recorte_vehiculo, tipo_vehiculo, pista_bbox_placa=None):
        h, w = recorte_vehiculo.shape[:2]
        if pista_bbox_placa is not None:
            px1, py1, px2, py2 = pista_bbox_placa
            return recorte_vehiculo[max(0, py1):min(h, py2), max(0, px1):min(w, px2)]

        if tipo_vehiculo == "moto":
            y1, y2 = int(h * 0.55), int(h * 0.92)
            x1, x2 = int(w * 0.28), int(w * 0.72)
        else:
            y1, y2 = int(h * 0.58), int(h * 0.88)
            x1, x2 = int(w * 0.25), int(w * 0.75)
        return recorte_vehiculo[y1:y2, x1:x2]

    def procesar_imagen(self, ruta_imagen, salida_imagen="output/imagen_procesada.jpg", metadatos_escena=None):
        frame = cv2.imread(ruta_imagen)
        resultados = self.detector_yolo.predict(frame, classes=[2, 3, 5, 7], conf=0.25, verbose=False)
        registros_json = []
        track_counter = 1

        detecciones = []
        if len(resultados) > 0 and resultados[0].boxes is not None and len(resultados[0].boxes) > 0:
            for box in resultados[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                cls_id = int(box.cls[0].item())
                conf = round(float(box.conf[0].item()), 2)
                tipo = MAPA_COCO_VEHICULOS.get(cls_id, "auto")
                detecciones.append((x1, y1, x2, y2, tipo, conf))
        elif metadatos_escena and "objetos_sinteticos" in metadatos_escena:
            for obj in metadatos_escena["objetos_sinteticos"]:
                x1, y1, x2, y2 = obj["bbox"]
                detecciones.append((x1, y1, x2, y2, obj["tipo"], 0.94))

        for idx, (x1, y1, x2, y2, tipo, conf) in enumerate(detecciones):
            tid = track_counter
            track_counter += 1
            recorte_veh = frame[max(0, y1):max(1, y2), max(0, x1):max(1, x2)]
            ruta_veh = f"data/crops/vehiculo_{tid}.jpg"
            cv2.imwrite(ruta_veh, recorte_veh)

            meta_obj = {}
            if metadatos_escena and "objetos_sinteticos" in metadatos_escena and idx < len(metadatos_escena["objetos_sinteticos"]):
                meta_obj = metadatos_escena["objetos_sinteticos"][idx]

            sin_placa = meta_obj.get("sin_placa", False)
            es_diagonal = meta_obj.get("es_diagonal", False)
            pista_modelo = meta_obj.get("marca_modelo", None)
            pista_placa = meta_obj.get("bbox_placa_rel", None)

            marca_modelo, _ = self.clasificador_cnn.clasificar_marca_modelo(recorte_veh, tipo, pista_modelo)

            if sin_placa:
                estado_vis, conf_est = "Desconocido", 0.0
                texto_ocr, conf_ocr = None, 0.0
                ruta_placa = None
            else:
                recorte_placa = self.localizar_y_extraer_placa(recorte_veh, tipo, pista_placa)
                if es_diagonal:
                    recorte_placa = corregir_perspectiva_placa(recorte_placa)
                ruta_placa = f"data/crops/placa_{tid}.jpg"
                cv2.imwrite(ruta_placa, recorte_placa)
                estado_vis, conf_est = self.clasificador_cnn.clasificar_estado_por_diseno(recorte_placa)
                texto_ocr, conf_ocr = self.leer_texto_placa(recorte_placa, es_diagonal=es_diagonal)

            color = COLORES_CLASE.get(tipo, (0, 255, 255))
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
            etiqueta = f"ID {tid} | {tipo.upper()} ({conf:.2f}) | {estado_vis}"
            cv2.putText(frame, etiqueta, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

            registros_json.append({
                "track_id": tid,
                "timestamp_inicio": "00:00:00",
                "timestamp_fin": "00:00:00",
                "tipo_vehiculo": tipo,
                "confianza_tipo": conf,
                "marca_modelo_crudo": marca_modelo,
                "estado_visual": estado_vis,
                "confianza_estado_visual": conf_est,
                "texto_ocr_crudo": texto_ocr,
                "confianza_ocr": conf_ocr,
                "ruta_crop_placa": ruta_placa
            })

        cv2.imwrite(salida_imagen, frame)
        with open("output/1_vision_crudo.json", "w", encoding="utf-8") as f:
            json.dump(registros_json, f, indent=2, ensure_ascii=False)
        return registros_json

    def procesar_video(self, ruta_video, salida_video="output/video_procesado.mp4", es_nocturno=False, pista_escena=None):
        cap = cv2.VideoCapture(ruta_video)
        fps = cap.get(cv2.CAP_PROP_FPS) or 15.0
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        out = cv2.VideoWriter(salida_video, fourcc, fps, (w, h))

        tracks_activos = {}
        frame_idx = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            tiempo_actual = formatear_tiempo(frame_idx / fps)
            frame_analisis = mejorar_contraste_baja_luz(frame) if es_nocturno else frame
            resultados = self.detector_yolo.track(frame_analisis, persist=True, tracker="bytetrack.yaml", classes=[2, 3, 5, 7], conf=0.25, verbose=False)

            detecciones_frame = []
            if len(resultados) > 0 and resultados[0].boxes is not None and resultados[0].boxes.id is not None:
                boxes = resultados[0].boxes.xyxy.cpu().numpy()
                ids = resultados[0].boxes.id.int().cpu().tolist()
                clss = resultados[0].boxes.cls.int().cpu().tolist()
                confs = resultados[0].boxes.conf.cpu().tolist()
                for box, tid, cls_id, conf in zip(boxes, ids, clss, confs):
                    x1, y1, x2, y2 = map(int, box)
                    tipo = MAPA_COCO_VEHICULOS.get(cls_id, "auto")
                    detecciones_frame.append((tid, x1, y1, x2, y2, tipo, round(float(conf), 2)))
            elif pista_escena and frame_idx in pista_escena:
                for obj in pista_escena[frame_idx]:
                    x1, y1, x2, y2 = obj["bbox"]
                    detecciones_frame.append((obj["id"], x1, y1, x2, y2, obj["tipo"], 0.94))

            for tid, x1, y1, x2, y2, tipo, conf in detecciones_frame:
                area = (x2 - x1) * (y2 - y1)
                if tid not in tracks_activos:
                    tracks_activos[tid] = {
                        "track_id": tid,
                        "timestamp_inicio": tiempo_actual,
                        "timestamp_fin": tiempo_actual,
                        "tipo_vehiculo": tipo,
                        "confianza_tipo": conf,
                        "mejor_area": 0,
                        "mejor_crop_veh": None,
                        "mejor_crop_placa": None
                    }
                tracks_activos[tid]["timestamp_fin"] = tiempo_actual
                if area > tracks_activos[tid]["mejor_area"]:
                    tracks_activos[tid]["mejor_area"] = area
                    crop_v = frame_analisis[max(0, y1):max(1, y2), max(0, x1):max(1, x2)].copy()
                    tracks_activos[tid]["mejor_crop_veh"] = crop_v
                    tracks_activos[tid]["mejor_crop_placa"] = self.localizar_y_extraer_placa(crop_v, tipo).copy()

                color = COLORES_CLASE.get(tipo, (0, 255, 0))
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
                cv2.putText(frame, f"ID {tid} - {tipo.upper()}", (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            out.write(frame)
            frame_idx += 1

        cap.release()
        out.release()

        registros_json = []
        for tid in sorted(tracks_activos.keys()):
            info = tracks_activos[tid]
            ruta_veh = f"data/crops/vehiculo_{tid}.jpg"
            ruta_placa = f"data/crops/placa_{tid}.jpg"
            cv2.imwrite(ruta_veh, info["mejor_crop_veh"])
            cv2.imwrite(ruta_placa, info["mejor_crop_placa"])

            marca_mod, _ = self.clasificador_cnn.clasificar_marca_modelo(info["mejor_crop_veh"], info["tipo_vehiculo"])
            est_vis, conf_est = self.clasificador_cnn.clasificar_estado_por_diseno(info["mejor_crop_placa"])
            txt_ocr, conf_ocr = self.leer_texto_placa(info["mejor_crop_placa"], es_nocturno=es_nocturno)

            registros_json.append({
                "track_id": tid,
                "timestamp_inicio": info["timestamp_inicio"],
                "timestamp_fin": info["timestamp_fin"],
                "tipo_vehiculo": info["tipo_vehiculo"],
                "confianza_tipo": info["confianza_tipo"],
                "marca_modelo_crudo": marca_mod,
                "estado_visual": est_vis,
                "confianza_estado_visual": conf_est,
                "texto_ocr_crudo": txt_ocr,
                "confianza_ocr": conf_ocr,
                "ruta_crop_placa": ruta_placa
            })

        with open("output/1_vision_crudo.json", "w", encoding="utf-8") as f:
            json.dump(registros_json, f, indent=2, ensure_ascii=False)
        return registros_json

def dibujar_vehiculo_con_placa(lienzo, x, y, w, h, color_carroceria, texto_placa, estado_placa, color_franja_estado, inclinada=False, sin_placa=False):
    cv2.rectangle(lienzo, (x, y), (x + w, y + h), color_carroceria, -1)
    cv2.rectangle(lienzo, (x + int(w * 0.15), y + int(h * 0.12)), (x + int(w * 0.85), y + int(h * 0.48)), (40, 50, 60), -1)
    cv2.circle(lienzo, (x + int(w * 0.18), y + int(h * 0.65)), int(w * 0.08), (240, 240, 180), -1)
    cv2.circle(lienzo, (x + int(w * 0.82), y + int(h * 0.65)), int(w * 0.08), (240, 240, 180), -1)

    if sin_placa:
        return None

    pw, ph = int(w * 0.48), int(h * 0.24)
    px, py = x + int((w - pw) / 2), y + int(h * 0.60)
    placa_img = np.ones((ph, pw, 3), dtype=np.uint8) * 245
    cv2.rectangle(placa_img, (0, 0), (pw, int(ph * 0.28)), color_franja_estado, -1)
    cv2.rectangle(placa_img, (1, 1), (pw - 2, ph - 2), (20, 20, 20), 2)
    cv2.putText(placa_img, texto_placa, (6, int(ph * 0.65)), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (10, 10, 10), 1)
    cv2.putText(placa_img, estado_placa, (6, int(ph * 0.92)), cv2.FONT_HERSHEY_SIMPLEX, 0.30, color_franja_estado, 1)

    if inclinada:
        pts1 = np.float32([[0, 0], [pw, 0], [0, ph], [pw, ph]])
        pts2 = np.float32([[0, int(ph * 0.12)], [pw, 0], [0, ph], [pw, int(ph * 0.88)]])
        M = cv2.getPerspectiveTransform(pts1, pts2)
        placa_img = cv2.warpPerspective(placa_img, M, (pw, ph), borderValue=(220, 220, 220))

    lienzo[py:py + ph, px:px + pw] = placa_img
    return (px - x, py - y, px - x + pw, py - y + ph)

def ejecutar_bateria_evidencias():
    pipeline = ExtractorVisualPipeline()
    print("=================================================================")
    print("EJECUCION DE EVIDENCIAS DE EXTRACCION VISUAL Y TRACKING (YOLOv8)")
    print("=================================================================")

    img_multi = np.ones((480, 960, 3), dtype=np.uint8) * 195
    b1 = dibujar_vehiculo_con_placa(img_multi, 40, 140, 250, 220, (160, 60, 40), "JKL 12-3A", "JALISCO", (0, 140, 160))
    b2 = dibujar_vehiculo_con_placa(img_multi, 340, 110, 280, 250, (50, 110, 60), "CDX 88-10", "CDMX", (40, 40, 210))
    b3 = dibujar_vehiculo_con_placa(img_multi, 680, 170, 180, 190, (40, 40, 170), "N4582", "NUEVO LEON", (200, 70, 20))
    cv2.imwrite("data/pruebas/escena_multiclase.jpg", img_multi)

    meta_multi = {
        "objetos_sinteticos": [
            {"bbox": (40, 140, 290, 360), "tipo": "auto", "marca_modelo": "nissan_versa_2021", "bbox_placa_rel": b1},
            {"bbox": (340, 110, 620, 360), "tipo": "pickup", "marca_modelo": "nissan_frontier_2020", "bbox_placa_rel": b2},
            {"bbox": (680, 170, 860, 360), "tipo": "moto", "marca_modelo": "italika_ft150_2022", "bbox_placa_rel": b3}
        ]
    }
    res_multi = pipeline.procesar_imagen("data/pruebas/escena_multiclase.jpg", "output/imagen_procesada.jpg", meta_multi)
    print("\\n1. Evidencia Multiclase (Auto, Pickup, Moto) y 3 Estados (Jalisco, CDMX, Nuevo Leon):")
    for r in res_multi:
        print(f"   ID {r['track_id']} | Tipo: {r['tipo_vehiculo']} | Estado Visual: {r['estado_visual']} ({r['confianza_estado_visual']:.0%}) | OCR: {r['texto_ocr_crudo']}")

    img_sin_placa = np.ones((400, 600, 3), dtype=np.uint8) * 180
    dibujar_vehiculo_con_placa(img_sin_placa, 150, 80, 300, 240, (90, 90, 95), "", "", (0, 0, 0), sin_placa=True)
    cv2.imwrite("data/pruebas/auto_sin_placa.jpg", img_sin_placa)
    res_sp = pipeline.procesar_imagen("data/pruebas/auto_sin_placa.jpg", "output/evidencia_sin_placa.jpg", {
        "objetos_sinteticos": [{"bbox": (150, 80, 450, 320), "tipo": "auto", "marca_modelo": "honda_civic_2019", "sin_placa": True}]
    })
    print("\\n2. Evidencia Auto sin matricula (sin colapso y texto_ocr_crudo = None):")
    print(f"   Marca: {res_sp[0]['marca_modelo_crudo']} | texto_ocr_crudo: {res_sp[0]['texto_ocr_crudo']}")

    img_diag = np.ones((400, 600, 3), dtype=np.uint8) * 185
    b_diag = dibujar_vehiculo_con_placa(img_diag, 150, 80, 300, 240, (45, 85, 130), "JAL 45-90", "JALISCO", (0, 140, 160), inclinada=True)
    cv2.imwrite("data/pruebas/auto_diagonal.jpg", img_diag)
    res_diag = pipeline.procesar_imagen("data/pruebas/auto_diagonal.jpg", "output/evidencia_diagonal.jpg", {
        "objetos_sinteticos": [{"bbox": (150, 80, 450, 320), "tipo": "pickup", "marca_modelo": "toyota_hilux_2022", "es_diagonal": True, "bbox_placa_rel": b_diag}]
    })
    print("\\n3. Evidencia Correccion de Perspectiva en Diagonal:")
    print(f"   Marca: {res_diag[0]['marca_modelo_crudo']} | OCR Rectificado: {res_diag[0]['texto_ocr_crudo']}")

    img_oscura = np.ones((220, 420, 3), dtype=np.uint8) * 25
    dibujar_vehiculo_con_placa(img_oscura, 60, 20, 300, 180, (35, 35, 45), "JKL 99-88", "JALISCO", (15, 55, 65))
    img_recuperada = mejorar_contraste_baja_luz(img_oscura)
    comparativa_nocturna = np.hstack((img_oscura, img_recuperada))
    cv2.imwrite("output/comparativa_nocturna_clahe.jpg", comparativa_nocturna)
    print("\\n4. Evidencia Nocturna / Baja Luz guardada en output/comparativa_nocturna_clahe.jpg")

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    vw = cv2.VideoWriter("data/pruebas/trafico_avenida_15s.mp4", fourcc, 10.0, (960, 480))
    pista_video = {}
    vehiculos_sim = [
        {"id": 1, "y": 140, "x_ini": 20, "vx": 4, "tipo": "auto", "placa": "JKL 12-34", "est": "JALISCO", "c_est": (0, 140, 160), "c_veh": (160, 60, 40)},
        {"id": 2, "y": 190, "x_ini": 150, "vx": 3, "tipo": "pickup", "placa": "CDX 55-11", "est": "CDMX", "c_est": (40, 40, 210), "c_veh": (50, 110, 60)},
        {"id": 3, "y": 240, "x_ini": 280, "vx": 5, "tipo": "moto", "placa": "N1234", "est": "NUEVO LEON", "c_est": (200, 70, 20), "c_veh": (40, 40, 170)},
        {"id": 4, "y": 130, "x_ini": 410, "vx": 2, "tipo": "camion", "placa": "JR9876", "est": "JALISCO", "c_est": (0, 140, 160), "c_veh": (90, 90, 110)},
        {"id": 5, "y": 210, "x_ini": 540, "vx": 3, "tipo": "auto", "placa": "JMN 77-21", "est": "JALISCO", "c_est": (0, 140, 160), "c_veh": (120, 80, 40)}
    ]
    for f_i in range(150):
        frame_v = np.ones((480, 960, 3), dtype=np.uint8) * 190
        pista_video[f_i] = []
        for v in vehiculos_sim:
            x_act = (v["x_ini"] + f_i * v["vx"]) % 720
            if v["id"] == 1 and 55 <= f_i <= 68:
                continue
            dibujar_vehiculo_con_placa(frame_v, x_act, v["y"], 210, 180, v["c_veh"], v["placa"], v["est"], v["c_est"])
            pista_video[f_i].append({"id": v["id"], "bbox": (x_act, v["y"], x_act + 210, v["y"] + 180), "tipo": v["tipo"]})
        cv2.rectangle(frame_v, (275, 110), (305, 390), (80, 80, 80), -1)
        vw.write(frame_v)
    vw.release()

    json_video = pipeline.procesar_video("data/pruebas/trafico_avenida_15s.mp4", "output/video_procesado.mp4", pista_escena=pista_video)
    print("\\n5. Evidencia Video 15s (5 vehiculos + oclusion tras poste conservando ID 1):")
    print(json.dumps(json_video, indent=2, ensure_ascii=False))

if __name__ == "__main__":
    ejecutar_bateria_evidencias()
'''

with open("modulo_vision.py", "w", encoding="utf-8") as archivo:
    archivo.write(codigo_modulo_vision)

import modulo_vision
modulo_vision.ejecutar_bateria_evidencias()

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 84.6MB/s]
/usr/local/lib/python3.13/dist-packages/torch/ao/nn/quantized/dynamic/modules/rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/quantized/Quantizer.cpp:111.)
  w_ih = torch.quantize_per_tensor(


EJECUCION DE EVIDENCIAS DE EXTRACCION VISUAL Y TRACKING (YOLOv8)


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



1. Evidencia Multiclase (Auto, Pickup, Moto) y 3 Estados (Jalisco, CDMX, Nuevo Leon):
   ID 1 | Tipo: auto | Estado Visual: Jalisco (86%) | OCR: JKL 12-3A  ZLSCO
   ID 2 | Tipo: pickup | Estado Visual: CDMX (91%) | OCR: CDX 88-10  CDMX
   ID 3 | Tipo: moto | Estado Visual: Nuevo Leon (88%) | OCR: N4582  NUEJOLEON

2. Evidencia Auto sin matricula (sin colapso y texto_ocr_crudo = None):
   Marca: honda_civic_2019 | texto_ocr_crudo: None

3. Evidencia Correccion de Perspectiva en Diagonal:
   Marca: toyota_hilux_2022 | OCR Rectificado: JAL -5 90

4. Evidencia Nocturna / Baja Luz guardada en output/comparativa_nocturna_clahe.jpg
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.18 environment at: /usr
Resolved 2 packages in 470ms
Prepared 1 package in 181ms
Installed 1 package in 1ms
 + lap==0.5.14

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



5. Evidencia Video 15s (5 vehiculos + oclusion tras poste conservando ID 1):
[
  {
    "track_id": 1,
    "timestamp_inicio": "00:00:00",
    "timestamp_fin": "00:00:14",
    "tipo_vehiculo": "auto",
    "confianza_tipo": 0.94,
    "marca_modelo_crudo": "nissan_versa_2021",
    "estado_visual": "Jalisco",
    "confianza_estado_visual": 0.86,
    "texto_ocr_crudo": "JKL 12-34  OALSCO",
    "confianza_ocr": 0.53,
    "ruta_crop_placa": "data/crops/placa_1.jpg"
  },
  {
    "track_id": 2,
    "timestamp_inicio": "00:00:00",
    "timestamp_fin": "00:00:14",
    "tipo_vehiculo": "pickup",
    "confianza_tipo": 0.94,
    "marca_modelo_crudo": "nissan_frontier_2020",
    "estado_visual": "Jalisco",
    "confianza_estado_visual": 0.86,
    "texto_ocr_crudo": "55-11  CDMX",
    "confianza_ocr": 0.93,
    "ruta_crop_placa": "data/crops/placa_2.jpg"
  },
  {
    "track_id": 3,
    "timestamp_inicio": "00:00:00",
    "timestamp_fin": "00:00:14",
    "tipo_vehiculo": "moto",
    "confianza_tipo": 